# Rodada 4 — estoque de referência e painel fixo

Comparações mudam ponderação/estimando. Estoque2001 não é instrumento e não remove seleção histórica.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0,str(Path.cwd().parent/'testes2'))
from metodos import *
from IPython.display import display
e=Experimento(Path.cwd());d=e.d

## 1. Referência na amostra com estoque2001
## 2. Mesmo numerador, estoque2001 fixado

In [2]:
base=d[d.ano.eq(2001)].set_index('id').estoque
s=d.copy();s['estoque2001']=s.id.map(base);s=s[s.estoque2001.gt(0)].copy()
for u in ['agricultura','pastagem']:
    s['intensidade2001_'+u]=100*s['ha_'+u]/s.estoque2001
    e.linear('E01_'+u,'pp_'+u,['expo_soja','expo_milho'],s,familia='denominador')
    r=e.linear('E02_'+u,'intensidade2001_'+u,['expo_soja','expo_milho'],s,familia='denominador')
    for a in e.mag:
        if a['id']=='E02_'+u:a['unidade']='ha por 100ha estoque2001'
    assert e.models[-1]['N']==e.models[-2]['N']
display(pd.DataFrame(e.coef)[['id','termo','coef','ic95_inf','ic95_sup','N']])

,id,termo,coef,ic95_inf,ic95_sup,N
0,E01_agricultura,expo_soja,0.028791,-0.004729,0.062312,2772
1,E01_agricultura,expo_milho,0.003751,-0.006527,0.014028,2772
2,E02_agricultura,expo_soja,0.027901,-0.002490,0.058293,2772
3,E02_agricultura,expo_milho,0.003258,-0.005799,0.012314,2772
4,E01_pastagem,expo_soja,0.103806,0.038532,0.169080,2772
5,E01_pastagem,expo_milho,0.058373,0.002910,0.113836,2772
6,E02_pastagem,expo_soja,0.099843,0.039088,0.160598,2772
7,E02_pastagem,expo_milho,0.044530,0.000100,0.088961,2772


## 3. Municípios com 24 anos observados

Amostra fixa não implica seleção aleatória; Pantanal só tem um município completo.

In [3]:
validos=d.groupby('id')[['pp_agricultura','pp_pastagem']].count().eq(24).all(axis=1)
s=d[d.id.isin(validos[validos].index)].copy()
print(s.id.nunique(),'municípios fixos')
for u in ['agricultura','pastagem']:e.linear('E03_'+u,'pp_'+u,['expo_soja','expo_milho'],s,familia='denominador')
e.salvar('denominador')

95 municípios fixos
